In [1]:
import pandas as pd
import requests
import os
from datetime import datetime

# paths
save_dir = "/lakehouse/default/Files/data/raw/bse_mcap"
log_file = "/lakehouse/default/Files/data/raw/logs/pipeline_log.csv"

os.makedirs(save_dir, exist_ok=True)
os.makedirs("/lakehouse/default/Files/data/raw/logs", exist_ok=True)

# today's date
today = datetime.today().strftime("%d-%m-%Y")

file_path = os.path.join(
    save_dir,
    f"bse_mcap_{today}.csv"
)

try:

    url = (
        "https://api.bseindia.com/BseIndiaAPI/api/"
        "ListofScripData/w"
        "?Group="
        "&Scripcode="
        "&segment=Equity"
        "&status="
    )

    headers = {
        "User-Agent": "Mozilla/5.0",
        "Referer": "https://www.bseindia.com/"
    }

    response = requests.get(
        url,
        headers=headers,
        timeout=60
    )

    response.raise_for_status()

    data = response.json()

    df = pd.DataFrame(data)

    bse_mcap_df = df[
        [
            "SCRIP_CD",
            "scrip_id",
            "Issuer_Name",
            "ISIN_NUMBER",
            "Mktcap"
        ]
    ].copy()

    bse_mcap_df["Mktcap"] = pd.to_numeric(
        bse_mcap_df["Mktcap"],
        errors="coerce"
    )

    bse_mcap_df = bse_mcap_df[
        bse_mcap_df["Mktcap"] > 0
    ]

    bse_mcap_df.to_csv(
        file_path,
        index=False
    )

    status = "SUCCESS"
    rows = len(bse_mcap_df)
    message = f"Saved: {os.path.basename(file_path)}"

except Exception as e:

    status = "FAILED"
    rows = 0
    message = str(e)

# log row
log_row = pd.DataFrame([{
    "timestamp": datetime.now(),
    "dataset": "bse_mcap",
    "status": status,
    "rows": rows,
    "message": message
}])

# append log
if os.path.exists(log_file):

    old_log = pd.read_csv(log_file)

    log_df = pd.concat(
        [old_log, log_row],
        ignore_index=True
    )

else:
    log_df = log_row

log_df.to_csv(
    log_file,
    index=False
)

print(status)
print("Rows:", rows)
print(message)

StatementMeta(, a8b2a693-bf59-4b55-8cf6-d454c9b9e5ce, 3, Finished, Available, Finished, False)

SUCCESS
Rows: 6532
Saved: bse_mcap_09-06-2026.csv
